# 06 — Creative Asset Visual Preview

Goal: visually inspect the synthetic PNG creative images alongside their performance metadata to build intuition about what the assets look like.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

DATA = Path("../")
ASSETS = DATA / "assets"

cs = pd.read_csv(DATA / "creative_summary.csv")

status_colors = {
    "top_performer": "#2ecc71",
    "stable": "#3498db",
    "fatigued": "#e67e22",
    "underperformer": "#e74c3c",
}


def load_image(creative_id):
    path = ASSETS / f"creative_{creative_id}.png"
    if path.exists():
        return Image.open(path)
    return None


print(f"Asset files found: {len(list(ASSETS.glob('*.png')))}")

## 1. Random Grid of 12 Creative Images

In [ ]:
sample_ids = cs.sample(12, random_state=42)["creative_id"].tolist()

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, cid in enumerate(sample_ids):
    img = load_image(cid)
    row = cs[cs["creative_id"] == cid].iloc[0]
    if img:
        axes[i].imshow(img)
    else:
        axes[i].text(
            0.5, 0.5, "Image not found", ha="center", va="center", transform=axes[i].transAxes
        )
    axes[i].set_title(
        f"ID {cid} | {row['creative_status']}\n{row['vertical']} · {row['format']}",
        fontsize=8,
        fontweight="bold",
        color=status_colors.get(row["creative_status"], "black"),
    )
    axes[i].axis("off")

plt.suptitle("Random Sample of 12 Creative Assets", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. One Example Per Creative Status (best perf_score in each class)

In [ ]:
status_order = ["top_performer", "stable", "fatigued", "underperformer"]

fig, axes = plt.subplots(1, 4, figsize=(18, 6))

for ax, status in zip(axes, status_order):
    best = cs[cs["creative_status"] == status].nlargest(1, "perf_score").iloc[0]
    cid = best["creative_id"]
    img = load_image(cid)
    if img:
        ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"{status.upper()}\n"
        f"ID {cid}\n"
        f"CTR: {best['overall_ctr']:.4f}\n"
        f"ROAS: {best['overall_roas']:.3f}\n"
        f"Theme: {best['theme']}\n"
        f"Format: {best['format']}",
        fontsize=8,
        fontweight="bold",
        color=status_colors[status],
    )

plt.suptitle(
    "Best-in-Class Creative by Status (highest perf_score)", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 3. Examples by Ad Format

In [ ]:
formats = cs["format"].unique()
n_formats = len(formats)

fig, axes = plt.subplots(1, n_formats, figsize=(5 * n_formats, 7))
if n_formats == 1:
    axes = [axes]

for ax, fmt in zip(axes, formats):
    example = cs[cs["format"] == fmt].nlargest(1, "perf_score").iloc[0]
    img = load_image(example["creative_id"])
    if img:
        ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"{fmt}\n"
        f"{example['width']}×{example['height']}\n"
        f"Duration: {example['duration_sec']}s\n"
        f"perf_score: {example['perf_score']:.3f}",
        fontsize=9,
        fontweight="bold",
    )

plt.suptitle(
    "Creative Examples by Ad Format (top perf_score per format)", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 4. Examples by Dominant Color

In [ ]:
colors_list = cs["dominant_color"].value_counts().head(8).index.tolist()
ncols = 4
nrows = (len(colors_list) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 6 * nrows))
axes = axes.flatten()

for i, color_name in enumerate(colors_list):
    example = cs[cs["dominant_color"] == color_name].nlargest(1, "perf_score").iloc[0]
    img = load_image(example["creative_id"])
    if img:
        axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(
        f"{color_name}\nCTR: {example['overall_ctr']:.4f} | Status: {example['creative_status']}",
        fontsize=9,
        fontweight="bold",
    )

for j in range(len(colors_list), len(axes)):
    axes[j].axis("off")

plt.suptitle(
    "Creative Examples by Dominant Color (top perf_score)", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 5. Top Performer vs Underperformer Side-by-Side (same vertical)

In [ ]:
# Find a vertical that has both top_performers and underperformers
for vertical in cs["vertical"].unique():
    has_top = (
        cs[(cs["vertical"] == vertical) & (cs["creative_status"] == "top_performer")].shape[0] > 0
    )
    has_under = (
        cs[(cs["vertical"] == vertical) & (cs["creative_status"] == "underperformer")].shape[0] > 0
    )
    if has_top and has_under:
        chosen_vertical = vertical
        break

print(f"Chosen vertical for comparison: {chosen_vertical}")

top_row = (
    cs[(cs["vertical"] == chosen_vertical) & (cs["creative_status"] == "top_performer")]
    .nlargest(1, "perf_score")
    .iloc[0]
)
under_row = (
    cs[(cs["vertical"] == chosen_vertical) & (cs["creative_status"] == "underperformer")]
    .nsmallest(1, "perf_score")
    .iloc[0]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 7))

for ax, row, label in [
    (axes[0], top_row, "TOP PERFORMER"),
    (axes[1], under_row, "UNDERPERFORMER"),
]:
    img = load_image(row["creative_id"])
    if img:
        ax.imshow(img)
    ax.axis("off")
    status = row["creative_status"]
    ax.set_title(
        f"{label} — {chosen_vertical}\n"
        f"Creative {row['creative_id']}\n"
        f"CTR: {row['overall_ctr']:.4f} | CVR: {row['overall_cvr']:.4f}\n"
        f"ROAS: {row['overall_roas']:.3f} | perf: {row['perf_score']:.3f}\n"
        f"Theme: {row['theme']} | Hook: {row['hook_type']}\n"
        f"Tone: {row['emotional_tone']} | Color: {row['dominant_color']}",
        fontsize=9,
        fontweight="bold",
        color=status_colors[status],
    )

plt.suptitle(
    f"Direct Comparison: Top Performer vs Underperformer ({chosen_vertical})",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 6. Image Size & Dimension Summary

In [ ]:
dim_summary = (
    cs.groupby(["width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
dim_summary["resolution"] = (
    dim_summary["width"].astype(str) + "×" + dim_summary["height"].astype(str)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(dim_summary["resolution"], dim_summary["count"], color="#8172B2", edgecolor="white")
ax.set_title("Creative Asset Resolutions (width × height)", fontweight="bold")
ax.set_ylabel("Number of Creatives")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

print(dim_summary.head(10).to_string(index=False))